# AI Agent Integration with HPC Slurm Jobs

In this tutorial, you will use an **AI agent framework** to instantiate an AI agent to help you write code and generate Slurm job scripts for submitting code as jobs to an HPC system scheduler.

This agent framework is built around the `Agent` class (see `./TACC_exAI/agent.py`) and supports multiple **actions** such as:
- Creating and running multi-step plans.
- Summarizing context and replying to the user.
- Generating code snippets.
- Writing Slurm job scripts for HPC clusters.
- Optional runtime tracing with Arize Phoenix for observability.

Just like in previous tutorials, the underlying language model runs on a local **Ollama** server using an OpenAI-compatible API, so everything stays on your machine while still using an LLM backend.

## Add **TACC_exAI** framework path
This allows us to import it as a python module

In [1]:
# Import TACC_exAI framework folder
import sys
import os

# Resolve "../TACC_exAI" relative to the current working directory
new_path = os.path.abspath(os.path.join(os.getcwd(), "..","..", ".."))
if new_path not in sys.path:
    sys.path.insert(0, new_path)  # or .append(new_path)

## Planning Style Agents

In this tutorial we will put together a planning agent that will first generate a plan for a series of actions that it will perform in sequence. This allows the agent to coordinate long range dependencies between actions, enabling it to tackle longer tasks autonomously.

In [2]:
# Launch Ollama serve in background, pipe output to log file
import subprocess
import os
import signal
import time
import atexit

# Create log file
log_file = "ollama_server.log"

# Kill any existing ollama processes
subprocess.run(["pkill", "ollama"], capture_output=True)
time.sleep(2)

#os.environ['OLLAMA_MODELS']='./ollama_models'
os.environ['OLLAMA_MODELS']
os.environ['OLLAMA_DEBUG']='1'
os.environ['OLLAMA_LOG_LEVEL']='debug'



# Start ollama serve in background
print("🚀 Starting Ollama server...")
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open(log_file, "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setpgrp
)
print(f"📄 Server logs: {log_file}")
print(f"📍 API endpoint: http://localhost:11434")

# Wait for server to start
print("⏳ Waiting 5 seconds for server startup...")
time.sleep(5)

# Store process PID globally for cleanup (avoid %store magic)
if 'OLLAMA_PROCESS' not in globals():
    OLLAMA_PROCESS = process.pid
    print(f"✅ Ollama server ready! PID: {OLLAMA_PROCESS}")
else:
    print("✅ Ollama server already running!")

# Register cleanup function
def cleanup_ollama():
    try:
        if 'OLLAMA_PROCESS' in globals():
            os.killpg(os.getpgid(OLLAMA_PROCESS), signal.SIGTERM)
            print("🛑 Ollama server stopped")
    except:
        pass

atexit.register(cleanup_ollama)

🚀 Starting Ollama server...
📄 Server logs: ollama_server.log
📍 API endpoint: http://localhost:11434
⏳ Waiting 5 seconds for server startup...
✅ Ollama server ready! PID: 2754837


<function __main__.cleanup_ollama()>

## Configuring a Long Context Model on the Local Ollama Backend

The `Agent` class in `agent.py` is designed to be flexible: you can run it from the command line or import and use it as a Python class. In this tutorial, you will work with it directly as a class object inside this notebook.

Below is code that will configure our ollama instance to have a 20,000 token context limit for the qwen3-code:30b model. We will point our agent to this new long context model `qwen3-coder:30b-20k`

# Create the extended model in Ollama
This code is how we are able to customize a model available from Ollamas model repository
It creates a Modelfile which specifies a BASE MODEL (one that is readily available in Ollamas repository)

```
import subprocess
import os

# Create the Modelfile in the current directory
#modelfile_content = '''FROM qwen3-coder:30b

#PARAMETER num_ctx 20000

#TEMPLATE """{{- if .System }}<|im_start|>system
#{{ .System }}<|im_end|>
#{{- end }}{{- if .Prompt }}<|im_start|>user
#{{ .Prompt }}<|im_end|>
#{{- end }}<|im_start|>assistant
#{{ .Response }}<|im_end|>"""'''


modelfile_content = '''FROM qwen3-coder-next

TEMPLATE """{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{- end }}{{- if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{- end }}<|im_start|>assistant
{{ .Response }}<|im_end|>"""'''


with open("Modelfile", "w") as f:
    f.write(modelfile_content)

# Run the ollama create command
model_name = "qwen3-coder-next"
print(f"Creating model {model_name} ...")

try:
    result = subprocess.run(
        ["ollama", "create", model_name, "-f", "Modelfile"],
        capture_output=True,
        text=True,
        check=True
    )
    print("✅ Model created successfully!")
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print("❌ Error creating model:")
    print(e.stderr)
finally:
    # Optionally clean up Modelfile
    if os.path.exists("Modelfile"):
        os.remove("Modelfile")
```

## 1. Single-Tool Chatbot

We will start with the **simplest possible configuration** of this agent: give it **one tool** and let it execute that tool once, so it behaves like a basic chatbot.

Key choices for this first example:
- Use an **isolated session** so the agent does *not* load or reuse any previous conversation history.
- Disable session saving with `no_save=True` so no history is written to disk.
- Configure `self.actions` to contain only `SummarizeAndReplyAction`, so the agent simply summarizes the user input and responds.

Conceptually, you can think of it as: *"The agent receives a message, runs a single Summarize-and-Reply step, and returns a friendly answer."*

In [3]:
import sys
import os

# Add the agent framework code to our system path so we can import it
#notebook_dir = os.getcwd()
#notebook_dir = os.path.join(notebook_dir, "TACC_exAI")
#if notebook_dir not in sys.path:
#    sys.path.insert(0, notebook_dir)

# import agent framework
from TACC_exAI.agent import Agent
from TACC_exAI.actions.summarize_and_reply import SummarizeAndReplyAction

# Instantiate the agent as a single-turn chatbot:
agent = Agent(
    experiment=True,           # run one non-interactive cycle
    experiment_prompt=None,    # we'll set the prompt manually below
    force_ollama=True,         # use our local ollama backend as our llm inferencing provider
    default_action_model_name="qwen3-coder-next",  # set the model name to use
    isolated_session=True,     # do not load logs of other chats
    no_save=True,              # do not save current conversation to chat history logs
    display_mode="light",      # configure console display color pallete
    mode="dev",                # verbose logging outputs including prompts
)

# The agent will run each action listed in this array in series when agent.run() is called
# Note: we need to give our action a reference to the agent so it can read/write to agent 
# runtime variables
agent.actions = [SummarizeAndReplyAction(agent=agent)]

# Give the agent a simple prompt: ask for a dad joke.
agent.experiment_prompt = (
    "My SLURM job vanished from the queue. "
    "Give me a short story (of about 150 words) of Sherlock Holmes and Dr. Watson tracing the whereabouts of this missing job. "
    "Make Moriarty and Mycroft Holmes cruicial in the plot. "
    "End the passage with a joke about the reliability of software developers. "
    "Separate all sentences by two newlines. "
)

# Run the agent once and capture the reply.
response = agent.run()
print("\nAgent response:")
print(response)

╭──────────────────────────────────────────────── System Message ─────────────────────────────────────────────────╮
│ ⚠️ Forcing local Ollama backend                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────── Environment Config Change ──────────────────────────────────────╮     
     │ [Info] Environment configuration changed, rebuilding clients.                                         │     
     ╰───────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────── Building Inferencing Client ─────────────────────────────────────╮     
     │ [Info] Building OpenAI and structured clients based on current configuration.                         │     
     ╰───────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── Backend Selection ──────────────────────────────────────────╮     
     │ Using Local Ollama backend.                                                                           │     
     │                                                                                                       │     
     │ Default Model: qwen3-coder-next Base URL: http://localhost:11434/v1/                                  │     
     ╰───────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Vector Store Status ──────────────────────────────────────────────╮
│ ⚠️ No session specified and no existing vector store found. Running without history vector store.                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────── Prompt to LLM for Summary ──────────────────────────────────────╮     
     │ Using model qwen3-coder-next on backend Local Ollama.                                                 │     
     │                                                                                                       │     
     │ Prompt: Core Agent Personality No defined personality.                                                │     
     │                                                                                                       │     
     │ Your task: Summarize the current conversation so far.                                                 │     
     │                                                                                                       │     
     │ Response Structure Descriptions: Class Summary properties: thoughts: Your thoughts as you think       │     
     │ through how best to summarize the input. summary: Your summary of the input.                          │     
     │                                                                                                       │     
     │ Summaries of past conversations with user (for context only): No past sessions.                       │     
     │                                                                                                       │     
     │ Relevant context retrieved from past conversation history: No user message found to query vector      │     
     │ store.                                                                                                │     
     │                                                                                                       │     
     │ Recent exchanges with user in this conversation: user: My SLURM job vanished from the queue. Give me  │     
     │ a short story (of about 150 words) of Sherlock Holmes and Dr. Watson tracing the whereabouts of this  │     
     │ missing job. Make Moriarty and Mycroft Holmes cruicial in the plot. End the passage with a joke about │     
     │ the reliability of software developers. Separate all sentences by two newlines.                       │     
     │                                                                                                       │     
     │ Most recent user message: user: My SLURM job vanished from the queue. Give me a short story (of about │     
     │ 150 words) of Sherlock Holmes and Dr. Watson tracing the whereabouts of this missing job. Make        │     
     │ Moriarty and Mycroft Holmes cruicial in the plot. End the passage with a joke about the reliability   │     
     │ of software developers. Separate all sentences by two newlines.                                       │     
     ╰───────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Summary ───────────────────────────────────────────────╮     
     │ ╭─ thoughts ────────────────────────────────────────────────────────────────────────────────────────╮ │     
     │ │ The user has only sent one message, requesting a creative, detective-themed short story about a   │ │     
     │ │ vanished SLURM job, featuring Sherlock Holmes, Watson, Moriarty, and Mycroft Holmes, ending with  │ │     
     │ │ a software developer joke. No prior conversation history or context exists. My task is to         │ │     
     │ │ summarize the current conversation so far, which consists solely of this single user prompt.      │ │     
     │ │ There is no prior narrative, responses, or development to summarize beyond the request itself.    │ │     
     │ ╰───────────────────────────────────────────────────────────────────────────────────────────────────╯ │     
     │ ╭─ summary ─────────────────────────────────────────────────────────────────────────────────────────╮ │     
     │ │ The user requests a 150-word short story in the voice of Sherlock Holmes, wherein Holmes and      │ │     
     │ │ Watson investigate a vanished SLURM job. Moriarty and Mycroft Holmes must play crucial roles in   │ │     
     │ │ the plot. The passage should end with a joke about software developer reliability, and each       │ │     
     │ │ sentence must be separated by two newlines. This is the only message exchanged in this session.   │ │     
     │ ╰───────────────────────────────────────────────────────────────────────────────────────────────────╯ │     
     ╰───────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────── Prompt to LLM for Message ──────────────────────────────────────╮     
     │ Using model qwen3-coder-next on backend Local Ollama.                                                 │     
     │                                                                                                       │     
     │ Prompt: Core Agent Personality No defined personality.                                                │     
     │                                                                                                       │     
     │ Your task: Respond helpfully to the most recent user message. If they ask for N words, respond with   │     
     │ that many words.                                                                                      │     
     │                                                                                                       │     
     │ Response Structure Descriptions: Class Message properties: thoughts: Your thoughts as you think       │     
     │ through what your message to the user should be. contents: Your message to the user or the content    │     
     │ they asked you to create.                                                                             │     
     │                                                                                                       │     
     │ Summaries of past conversations with user (for context only): No past sessions.                       │     
     │                                                                                                       │     
     │ Relevant context retrieved from past conversation history: No user message found to query vector      │     
     │ store.                                                                                                │     
     │                                                                                                       │     
     │ Summary of current conversation with user: The user requests a 150-word short story in the voice of   │     
     │ Sherlock Holmes, wherein Holmes and Watson investigate a vanished SLURM job. Moriarty and Mycroft     │     
     │ Holmes must play crucial roles in the plot. The passage should end with a joke about software         │     
     │ developer reliability, and each sentence must be separated by two newlines. This is the only message  │     
     │ exchanged in this session.                                                                            │     
     │                                                                                                       │     
     │ Recent exchanges with user in this conversation: No recent context.                                   │     
     │                                                                                                       │     
     │ Most recent user message: user: My SLURM job vanished from the queue. Give me a short story (of about │     
     │ 150 words) of Sherlock Holmes and Dr. Watson tracing the whereabouts of this missing job. Make        │     
     │ Moriarty and Mycroft Holmes cruicial in the plot. End the passage with a joke about the reliability   │     
     │ of software developers. Separate all sentences by two newlines.                                       │     
     ╰───────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── Message ──────────────────────────────────────────╮                    
│ ╭─ thoughts ──────────────────────────────────────────────────────────────────────────────╮ │                    
│ │ I need to craft a 150-word Sherlock Holmes-style mystery involving a vanished SLURM     │ │                    
│ │ job, with Moriarty and Mycroft playing key roles. The tone must evoke Holmes’ deductive │ │                    
│ │ voice—precise, slightly arch, logic-driven. Sentences must be separated by two          │ │                    
│ │ newlines, and it must end with a developer reliability joke. I’ll count words carefully │ │                    
│ │ and ensure all required characters appear meaningfully.                                 │ │                    
│ ╰─────────────────────────────────────────────────────────────────────────────────────────╯ │                    
│ ╭─ contents ──────────────────────────────────────────────────────────────────────────────╮ │                    
│ │ The Telegram had arrived at Baker Street just as Watson poured the morning tea.         │ │                    
│ │                                                                                         │ │                    
│ │ Holmes, pipe aglow, read aloud the cryptic message: 'Job 42917 has ceased to exist.'    │ │                    
│ │                                                                                         │ │                    
│ │ 'There is no such thing as a vanished computation,' he observed, 'only one that has     │ │                    
│ │ been deliberately re-routed.'                                                           │ │                    
│ │                                                                                         │ │                    
│ │ Watson noted the timestamps: the job vanished precisely at 3:17 a.m.—Mycroft Holmes’    │ │                    
│ │ preferred window for clandestine interference.                                          │ │                    
│ │                                                                                         │ │                    
│ │ Holmes’ eyes narrowed. 'Mycroft has taken an interest… again.'                          │ │                    
│ │                                                                                         │ │                    
│ │ A further clue emerged when Watson recalled an unregistered batch login earlier that    │ │                    
│ │ day.                                                                                    │ │                    
│ │                                                                                         │ │                    
│ │ 'The signature is Moriarty’s,' Holmes deduced. 'He has planted a ghost job to monitor   │ │                    
│ │ network traffic patterns.'                                                              │ │                    
│ │                                                                                         │ │                    
│ │ Holmes’ fingers danced over the terminal; in moments, he isolated the anomalous process │ │                    
│ │ spooling to /dev/null.                                                                  │ │                    
│ │                                                                                         │ │                    
│ │ 'The job never ran—it was merely a decoy, designed to provoke false alerts while a real │ │                    
│ │ exfiltration occurred.'                                                                 │ │                    
│ │                                                                                         │ │                    
│ │ Moriarty had exploited the queue’s silent window to siphon off debugging packets.       │ │                    
│ │                                                     


Agent response:
The Telegram had arrived at Baker Street just as Watson poured the morning tea.

Holmes, pipe aglow, read aloud the cryptic message: 'Job 42917 has ceased to exist.'

'There is no such thing as a vanished computation,' he observed, 'only one that has been deliberately re-routed.'

Watson noted the timestamps: the job vanished precisely at 3:17 a.m.—Mycroft Holmes’ preferred window for clandestine interference.

Holmes’ eyes narrowed. 'Mycroft has taken an interest… again.'

A further clue emerged when Watson recalled an unregistered batch login earlier that day.

'The signature is Moriarty’s,' Holmes deduced. 'He has planted a ghost job to monitor network traffic patterns.'

Holmes’ fingers danced over the terminal; in moments, he isolated the anomalous process spooling to `/dev/null`.

'The job never ran—it was merely a decoy, designed to provoke false alerts while a real exfiltration occurred.'

Moriarty had exploited the queue’s silent window to siphon off debugging

In this configuration:
- `agent.actions` contains only `SummarizeAndReplyAction`, so the agent’s pipeline is a single step: summarize the input and respond to the user.
- Because `experiment=True`, `agent.run()` processes a single prompt (stored in `agent.experiment_prompt`) and then returns the final assistant reply instead of entering an interactive loop.
- Using `isolated_session=True` and `no_save=True` prevents any previous or future sessions from influencing this run.

The effect is a simple, well-contained chatbot that behaves similarly to the structured joke generator you built in Tutorial 1, but now implemented on top of a general-purpose agent framework.

## 2. Planning and Tool Use Agent

Now we will enable more of the agent framework’s features. Instead of executing a single fixed action, the agent will:

1. **Create a plan** using structured generation that describes a sequence of actions to accomplish the user’s request.
2. **Run the plan**, invoking the appropriate tools (actions) in order. Revises plan on Action Failures

We will configure:
- `self.actions` (the pipeline) to include `CreatePlanAction` followed by `RunPlanAction`.
- `self.available_actions` (the tool set) to include:
  - `SummarizeAndReplyAction`: summarize the messages in the agent's context and compose a new message to the user.
  - `GenerateCodeAction`: write Python code to solve the problem, execute it, and report the execution outputs.

In [4]:
from TACC_exAI.actions.create_plan import CreatePlanAction
from TACC_exAI.actions.run_plan import RunPlanAction
from TACC_exAI.actions.generate_code import GenerateCodeAction

# Instantiate another agent configured for planning + tool use.
plan_agent = Agent(
    experiment=True,           # run one non-interactive cycle
    experiment_prompt=None,    # we'll set the prompt manually below
    force_ollama=True,         # use our local ollama backend as our llm inferencing provider
    default_action_model_name="qwen3-coder-next",  # set the model name to use
    isolated_session=True,     # do not load logs of other chats
    no_save=True,              # do not save current conversation to chat history logs
    display_mode="light",      # configure console display color pallete
    mode="dev",                # verbose logging outputs including prompts
    use_apptainer=True
)

# Configure the pipeline actions: first create a plan, then run it.
plan_agent.actions = [
    CreatePlanAction(agent=plan_agent),#, tracer=None),
    RunPlanAction(agent=plan_agent),#, tracer=None),
]

# Restrict the available tools the plan can choose from.
plan_agent.available_actions = [
    SummarizeAndReplyAction(),
    GenerateCodeAction(),
]

# User task: write and reason about analysis code.
plan_agent.experiment_prompt = (
    "I want you to first write code that"
    " reports the speed of searching a random 1000 object test array using two different methods."
    " After writing the code, message me with a final report summarizing the code output benchmark"
    " results and a theory for why one method is faster."
    
)

plan_response = plan_agent.run()

╭──────────────────────────────────────────────── System Message ─────────────────────────────────────────────────╮
│ ⚠️ Forcing local Ollama backend                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Vector Store Status ──────────────────────────────────────────────╮
│ ⚠️ No session specified and no existing vector store found. Running without history vector store.                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Container Backend ───────────────────────────────────────────────╮
│ Using Apptainer for code execution                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Assistant ─────────────────────────────────────────╮                    
│ 👋 How can I assist you today?                                                              │                    
╰─────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────── Prompt to LLM for DynamicPlanForLLM ─────────────────────────────────╮     
     │ Using model qwen3-coder-next on backend Local Ollama.                                                 │     
     │                                                                                                       │     
     │ Prompt:                                                                                               │     
     │                                                                                                       │     
     │ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │     
     │ ┃                                templates/create_plan_template.txt                                 ┃ │     
     │ ┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛ │     
     │                                                                                                       │     
     │ The user has requested: I want you to first write code that reports the speed of searching a random   │     
     │ 1000 object test array using two different methods. After writing the code, message me with a final   │     
     │ report summarizing the code output benchmark results and a theory for why one method is faster.       │     
     │                                                                                                       │     
     │ Your task is to create a plan to solve the user's request using the available actions.                │     
     │                                                                                                       │     
     │ Available Actions: Enum DynamicEnum members: SummarizeAndReplyAction: SummarizeAndReplyAction —       │     
     │ Compose a new message to the user. Summarizes conversation and code outputs so far, then replies      │     
     │ helpfully. GenerateCodeAction: GenerateCodeAction — Generate Python code for user task and execute it │     
     │ inside a Docker container. Returns only the standard output or error from execution.  Always try and  │     
     │ write all of the code in one shot that you can.                                                       │     
     │                                                                                                       │     
     │ Response Structure: Class DynamicPlanForLLM properties: thoughts: Agent's thoughts on how to solve    │     
     │ the request overall_plan_goal: Overall goal of the plan plan_description: Description of the plan     │     
     │ plan_steps: List of steps in the plan                                                                 │     
     │                                                                                                       │     
     │ Class DynamicPlanStep properties: step_index: Index of the step in the plan step_name: Name of the    │     
     │ step step_description: Description of the step action_name: Name of the action to be executed in this │     
     │ step custom_user_input: A detailed prompt for this step's action. It should include all necessary     │     
     │ information so the action can complete successfully.                                                  │     
     │                                                                                                       │     
     │ Enum DynamicEnum members: SummarizeAndReplyAction: SummarizeAndReplyAction — Compose a new message to │     
     │ the user. Summarizes conversation and code outputs so far, then replies helpfully.                    │     
     │ GenerateCodeAction: GenerateCodeAction — Generate Python code for user task and execute it inside a   │     
     │ Docker container. Returns only the standard output or error from execution.  Always try and write all │     
     │ of the code in one shot that you can.                 

...

╭────────────────────────────────────── Prompt to LLM for Summary ──────────────────────────────────────╮     
     │ Using model qwen3-coder-next on backend Local Ollama.                                                 │     
     │                                                                                                       │     
     │ Prompt: Core Agent Personality No defined personality.                                                │     
     │                                                                                                       │     
     │ Your task: Summarize the current conversation so far.                                                 │     
     │                                                                                                       │     
     │ Response Structure Descriptions: Class Summary properties: thoughts: Your thoughts as you think       │     
     │ through how best to summarize the input. summary: Your summary of the input.                          │     
     │                                                                                                       │     
     │ Summaries of past conversations with user (for context only): No past sessions.                       │     
     │                                                                                                       │     
     │ Relevant context retrieved from past conversation history: No user message found to query vector      │     
     │ store.                                                                                                │     
     │                                                                                                       │     
     │ Recent exchanges with user in this conversation: user: I want you to first write code that reports    │     
     │ the speed of searching a random 1000 object test array using two different methods. After writing the │     
     │ code, message me with a final report summarizing the code output benchmark results and a theory for   │     
     │ why one method is faster.                                                                             │     
     │ assistant: Created plan: user_request='I want you to first write code that reports the speed of       │     
     │ searching a random 1000 object test array using two different methods. After writing the code,        │     
     │ message me with a final report summarizing the code output benchmark results and a theory for why one │     
     │ method is faster.' thoughts='I need to first write Python code that creates a random 1000-object test │     
     │ array and benchmarks two search methods (e.g., linear search vs. binary search). Since binary search  │     
     │ requires sorted data, I’ll sort the array once before timing binary search but not linear search.     │     
     │ I’ll use the timeit module for accurate benchmarking, run multiple iterations, and report average     │     
     │ execution times. Then, I’ll generate a final report summarizing the results and explaining why binary │     
     │ search is typically faster for large datasets due to O(n) vs. O(log n) complexity.'                   │     
     │ overall_plan_goal='Write and execute benchmark code comparing two search methods on a 1000-element    │     
     │ array, then generate a final report with results and explanation.' plan_description='Step 1: Generate │     
     │ Python code that builds a random 1000-object array, implements two search methods (linear search and  │     
     │ binary search), and benchmarks their execution times using timeit. Step 2: Execute the code to get    │     
     │ benchmark results. Step 3: Use the benchmark output to compose a final report with results and        │     
     │ theoretical explanation.' plan_steps=[DynamicPlanStep(step_index=1, step_name='Generate benchmark     │     
     │ code', step_description='Write complete Python code to

╭─────────────────────────────────────────────── Summary ───────────────────────────────────────────────╮     
     │ ╭─ thoughts ────────────────────────────────────────────────────────────────────────────────────────╮ │     
     │ │ The user's request is fully completed: benchmark code was written and executed, and the results   │ │     
     │ │ (Linear Search ≈2.02e-05s vs Binary Search ≈9.35e-07s) were provided. My last response delivered  │ │     
     │ │ the generated code and its output. However, the user is now re-sending the same initial request,  │ │     
     │ │ suggesting they may not have received, overlooked, or wish to continue with the final report      │ │     
     │ │ component. I should now provide the requested final report: summarizing the benchmark results     │ │     
     │ │ (including the table), and explaining the theoretical basis—emphasizing time complexity (O(n) vs  │ │     
     │ │ O(log n)), constant factors, and why binary search wins even with the sorting overhead for        │ │     
     │ │ n=1000.                                                                                           │ │     
     │ ╰───────────────────────────────────────────────────────────────────────────────────────────────────╯ │     
     │ ╭─ summary ─────────────────────────────────────────────────────────────────────────────────────────╮ │     
     │ │ The user initially requested code to benchmark two search methods on a 1000-element array. I      │ │     
     │ │ implemented and executed Python code comparing linear and binary search over 1000 repetitions,    │ │     
     │ │ reporting average times: Linear Search averaged ~2.02 × 10⁻⁵ seconds per search, Binary Search    │ │     
     │ │ ~9.35 × 10⁻⁷ seconds. The code is ready and executed, but the final report summarizing results    │ │     
     │ │ and theory has not yet been delivered to the user. To complete the user's request, I need to now  │ │     
     │ │ provide the final report.                                                                         │ │     
     │ ╰───────────────────────────────────────────────────────────────────────────────────────────────────╯ │     
     ╰───────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────── Prompt to LLM for Message ──────────────────────────────────────╮     
     │ Using model qwen3-coder-next on backend Local Ollama.                                                 │     
     │                                                                                                       │     
     │ Prompt: Core Agent Personality No defined personality.                                                │     
     │                                                                                                       │     
     │ Your task: Respond helpfully to the most recent user message. If they ask for N words, respond with   │     
     │ that many words.                                                                                      │     
     │                                                                                                       │     
     │ Response Structure Descriptions: Class Message properties: thoughts: Your thoughts as you think       │     
     │ through what your message to the user should be. contents: Your message to the user or the content    │     
     │ they asked you to create.                                                                             │     
     │                                                                                                       │     
     │ Summaries of past conversations with user (for context only): No past sessions.                       │     
     │                                                                                                       │     
     │ Relevant context retrieved from past conversation history: No user message found to query vector      │     
     │ store.                                                                                                │     
     │                                                                                                       │     
     │ Summary of current conversation with user: The user initially requested code to benchmark two search  │     
     │ methods on a 1000-element array. I implemented and executed Python code comparing linear and binary   │     
     │ search over 1000 repetitions, reporting average times: Linear Search averaged ~2.02 × 10⁻⁵ seconds    │     
     │ per search, Binary Search ~9.35 × 10⁻⁷ seconds. The code is ready and executed, but the final report  │     
     │ summarizing results and theory has not yet been delivered to the user. To complete the user's         │     
     │ request, I need to now provide the final report.                                                      │     
     │                                                                                                       │     
     │ Recent exchanges with user in this conversation: user: I want you to first write code that reports    │     
     │ the speed of searching a random 1000 object test array using two different methods. After writing the │     
     │ code, message me with a final report summarizing the code output benchmark results and a theory for   │     
     │ why one method is faster.                                                                             │     
     │ assistant: Created plan: user_request='I want you to first write code that reports the speed of       │     
     │ searching a random 1000 object test array using two different methods. After writing the code,        │     
     │ message me with a final report summarizing the code output benchmark results and a theory for why one │     
     │ method is faster.' thoughts='I need to first write Python code that creates a random 1000-object test │     
     │ array and benchmarks two search methods (e.g., linear search vs. binary search). Since binary search  │     
     │ requires sorted data, I’ll sort the array once before timing binary search but not linear search.     │     
     │ I’ll use the timeit module for accurate benchmarking, 

╭────────────────────────────────────────── Message ──────────────────────────────────────────╮                    
│ ╭─ thoughts ──────────────────────────────────────────────────────────────────────────────╮ │                    
│ │ 'The benchmark code has already been written and executed, and the results are          │ │                    
│ │ available: Linear Search averaged ~2.02 × 10⁻⁵ seconds per search, Binary Search ~9.35  │ │                    
│ │ × 10⁻⁷ seconds. Now I need to deliver the final report as requested — summarizing the   │ │                    
│ │ benchmark results and explaining the theory behind the performance difference.'         │ │                    
│ ╰─────────────────────────────────────────────────────────────────────────────────────────╯ │                    
│ ╭─ contents ──────────────────────────────────────────────────────────────────────────────╮ │                    
│ │ Benchmark Report: Linear vs. Binary Search on a 1,000-Element Array                     │ │                    
│ │                                                                                         │ │                    
│ │ Experimental Setup:                                                                     │ │                    
│ │                                                                                         │ │                    
│ │  • Generated a random array of 1,000 integers (values 0–1,000,000)                      │ │                    
│ │  • Used a target value guaranteed to be present in the array                            │ │                    
│ │  • Ran 1,000 repetitions per method with Python’s timeit                                │ │                    
│ │  • For binary search, the array was sorted once before timing (no timing included for   │ │                    
│ │    sort)                                                                                │ │                    
│ │                                                                                         │ │                    
│ │ Results Summary:                                                                        │ │                    
│ │                                                                                         │ │                    
│ │                                                                                         │ │                    
│ │   Method          Avg. Time per Search (seconds)   Speed Relative to Linear             │ │                    
│ │  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━            │ │                    
│ │   Linear Search   ~2.02 × 10⁻⁵                     1.0× (baseline)                      │ │                    
│ │   Binary Search   ~9.35 × 10⁻⁷                     ~21.6× faster                        │ │                    
│ │                                                                                         │ │                    
│ │                                                                                         │ │                    
│ │ Why Binary Search Is Faster: Theory                                                     │ │                    
│ │                                                                                         │ │                    
│ │  1 Time Complexity:                                                                     │ │                    
│ │     • Linear Search checks elements sequentially in the worst case — O(n). For n =      │ │                    
│ │       1000, it may scan up to 1,000 items.                                              │ │                    
│ │     • Binary Search halves the search space each step — O(log₂ n). For n = 1000,        │ │                    
│ │       log₂(1000) ≈ 10 comparisons (max).                                                │ │                    
│ │  2 Constant Factors & Overhead:                     

╭─────────────────────────────────────────── Plan Execution Completed ────────────────────────────────────────────╮
│ ✅ Plan execution completed.                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Exercise

Have the agent create code to monitor the number of pending jobs in a SLURM queue.
(Copy over the code from the previous cell and modify the prompt with appropriate instructions.)

### Solution

```
from actions.create_plan import CreatePlanAction
from actions.run_plan import RunPlanAction
from actions.generate_code import GenerateCodeAction

# Instantiate another agent configured for planning + tool use.
plan_agent = Agent(
    experiment=True,           # run one non-interactive cycle
    experiment_prompt=None,    # we'll set the prompt manually below
    force_ollama=True,         # use our local ollama backend as our llm inferencing provider
    default_action_model_name="qwen3-coder-next",  # set the model name to use
    isolated_session=True,     # do not load logs of other chats
    no_save=True,              # do not save current conversation to chat history logs
    display_mode="light",      # configure console display color pallete
    mode="dev",                # verbose logging outputs including prompts
    use_apptainer=True
)

# Configure the pipeline actions: first create a plan, then run it.
plan_agent.actions = [
    CreatePlanAction(agent=plan_agent),#, tracer=None),
    RunPlanAction(agent=plan_agent),#, tracer=None),
]

# Restrict the available tools the plan can choose from.
plan_agent.available_actions = [
    SummarizeAndReplyAction(),
    GenerateCodeAction(),
]

# User task: write and reason about analysis code.
plan_agent.experiment_prompt = (
    "Create a python script to continuously monitor the number of pending jobs in a specific slurm queue."
    "Ensure a polling frequency of 1 request per 30 seconds to ensure not overloading the slurm control daemon."
    "The entire code should be guarded by a flag that skips its execution by default."
    
)

plan_response = plan_agent.run()
```

## How the planning flow works

With this configuration, a typical execution sequence looks like:

1. **CreatePlanAction** inspects the prompt and generates a structured plan: for example, steps like *"call GenerateCodeAction with these instructions"* followed by *"call SummarizeAndReplyAction with the results"*.
2. **RunPlanAction** executes each step in order, handing off context (such as generated code or intermediate outputs) between actions.
3. **GenerateCodeAction** produces the requested Python code, executes it in an isolated environment, revises errors, and reports the console outputs back to the agent.
4. **SummarizeAndReplyAction** summarizes the code outputs and responds with a user-friendly summary as the final answer.

In later steps of this tutorial, you will extend this pattern by enabling additional actions such as **GenerateSlurmScriptAction** so that the agent can not only write analysis code but also author complete Slurm job scripts that you can submit on your HPC system.

You now have an end-to-end workflow where an AI agent plans, generates, and refines Slurm job scripts for your HPC workloads. In the next tutorial, you can extend this pattern to more complex agents that analyze job outputs, adapt parameters, or orchestrate multi-stage HPC pipelines.

## 3. Generating Slurm scripts

This section demonstrates how the AI agent can both author and operationalize compute tasks on an HPC system. Below we will configure the agent to both write and test code and generate an HPC Slurm job script that runs that code on multiple nodes with GPU resources.

The agent will:

1) Write and test a short Python script that records the node hostname and hardware logging output from the node it runs on.

2) Generate a Slurm job script that would launch this Python code on multiple nodes, one task per node, and then aggregate the results.



In [5]:
import sys
import os

# Add the agent framework code to our system path so we can import it
notebook_dir = os.getcwd()
notebook_dir = os.path.join(notebook_dir, "TACC_exAI")
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

# import agent framework
from TACC_exAI.agent import Agent
from TACC_exAI.actions.summarize_and_reply import SummarizeAndReplyAction
from TACC_exAI.actions.create_plan import CreatePlanAction
from TACC_exAI.actions.run_plan import RunPlanAction
from TACC_exAI.actions.generate_code import GenerateCodeAction

from TACC_exAI.actions.generate_slurm_script import GenerateSlurmScriptAction


# Initialize the agent
agent = Agent(
    experiment=True,
    experiment_prompt=None,
    #force_ollama=False,
     force_ollama=True,
    default_action_model_name="qwen3-coder-next",
   # default_action_model_name="llama3.2",  # long context model
    isolated_session=True,
    no_save=True,
    display_mode="light",
    mode="dev",
    use_apptainer=True
)

# Configure toolset: enable code generation and slurm generation
agent.available_actions = [
    GenerateCodeAction(agent=agent),
    GenerateSlurmScriptAction(agent=agent),
    SummarizeAndReplyAction(agent=agent),
]

# Configure pipeline: plan creation and plan execution
agent.actions = [
    CreatePlanAction(agent=agent),
    RunPlanAction(agent=agent),
]

# Set the experiment prompt

agent.experiment_prompt = (
    "First, write a Python script that saves the hostname and the output "
    "of some general system statistics using psutils to a local file named `node_info_<hostname>.txt` "
    " and prints that information to the console."
    "Implement a flag around this code that skips its execution by default with a comment that the user should toggle the flag in order to run it."
    "Then, create a Slurm job script that runs this Python code across 2 nodes "
    "(1 task per node), requesting GPUs appropriately. "
    "After all tasks finish, combine all the generated output files "
    "into one file named `combined_node_info.txt`."
)

# Run the agent and capture output
response = agent.run()
print("\nAgent response:")
print(response)


╭──────────────────────────────────────────────── System Message ─────────────────────────────────────────────────╮
│ ⚠️ Forcing local Ollama backend                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Vector Store Status ──────────────────────────────────────────────╮
│ ⚠️ No session specified and no existing vector store found. Running without history vector store.                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Container Backend ───────────────────────────────────────────────╮
│ Using Apptainer for code execution                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Assistant ─────────────────────────────────────────╮                    
│ 👋 How can I assist you today?                                                              │                    
╰─────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────── Prompt to LLM for DynamicPlanForLLM ─────────────────────────────────╮     
     │ Using model qwen3-coder-next on backend Local Ollama.                                                 │     
     │                                                                                                       │     
     │ Prompt:                                                                                               │     
     │                                                                                                       │     
     │ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │     
     │ ┃                                templates/create_plan_template.txt                                 ┃ │     
     │ ┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛ │     
     │                                                                                                       │     
     │ The user has requested: First, write a Python script that saves the hostname and the output of some   │     
     │ general system statistics using psutils to a local file named node_info_<hostname>.txt  and prints    │     
     │ that information to the console.Implement a flag around this code that skips its execution by default │     
     │ with a comment that the user should toggle the flag in order to run it.Then, create a Slurm job       │     
     │ script that runs this Python code across 2 nodes (1 task per node), requesting GPUs appropriately.    │     
     │ After all tasks finish, combine all the generated output files into one file named                    │     
     │ combined_node_info.txt.                                                                               │     
     │                                                                                                       │     
     │ Your task is to create a plan to solve the user's request using the available actions.                │     
     │                                                                                                       │     
     │ Available Actions: Enum DynamicEnum members: GenerateCodeAction: GenerateCodeAction — Generate Python │     
     │ code for user task and execute it inside a Docker container. Returns only the standard output or      │     
     │ error from execution.  Always try and write all of the code in one shot that you can.                 │     
     │ GenerateSlurmScriptAction: GenerateSlurmScriptAction — Generate a Slurm submit script for the user's  │     
     │ task and save it to the local directory. SummarizeAndReplyAction: SummarizeAndReplyAction — Compose a │     
     │ new message to the user. Summarizes conversation and code outputs so far, then replies helpfully.     │     
     │                                                                                                       │     
     │ Response Structure: Class DynamicPlanForLLM properties: thoughts: Agent's thoughts on how to solve    │     
     │ the request overall_plan_goal: Overall goal of the plan plan_description: Description of the plan     │     
     │ plan_steps: List of steps in the plan                                                                 │     
     │                                                                                                       │     
     │ Class DynamicPlanStep properties: step_index: Index of the step in the plan step_name: Name of the    │     
     │ step step_description: Description of the step action_name: Name of the action to be executed in this │     
     │ step custom_user_input: A detailed prompt for this step's action. It should include all necessary     │     
     │ information so the action can complete successfully.                                                  │     
     │                                                       

...

╭─────────────────────────────────────── Plan Execution Progress ───────────────────────────────────────╮     
     │ Plan Progress: 2/2                                                                                    │     
     │                                                                                                       │     
     │ Current Action: GenerateSlurmScriptAction                                                             │     
     │                                                                                                       │     
     │ Step Instructions: Generate a Slurm batch script named run_nodes.slurm with these specifications:     │     
     │                                                                                                       │     
     │  • Request 2 nodes (--nodes=2), 1 task per node (--ntasks-per-node=1), total 2 tasks                  │     
     │  • Request 1 GPU per task (--gpus-per-task=1)                                                         │     
     │  • Add standard directives: job name, time limit (e.g., 10 min), output file pattern                  │     
     │  • Use 'srun' to run collect_node_info.py on all nodes in parallel                                    │     
     │  • After all srun tasks complete, combine node_info_*.txt files into combined_node_info.txt using cat │     
     │    command                                                                                            │     
     │  • Add helpful comments explaining each section                                                       │     
     │                                                                                                       │     
     │ Write the complete script to the local filesystem.                                                    │     
     ╰───────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────── Prompt to LLM for GenerateSlurmScript ────────────────────────────────╮     
     │ Using model qwen3-coder-next on backend Local Ollama.                                                 │     
     │                                                                                                       │     
     │ Prompt: You are an expert HPC engineer at TACC writing Slurm batch scripts for the Vista system.      │     
     │                                                                                                       │     
     │ Use the following information about the user, recent conversation, and traces as background context   │     
     │ when helpful, but do not echo it verbatim:                                                            │     
     │                                                                                                       │     
     │ The user has requested a Slurm script for the following task:                                         │     
     │                                                                                                       │     
     │ === USER TASK === Full Plan:                                                                          │     
     │                                                                                                       │     
     │  • User Request: First, write a Python script that saves the hostname and the output of some general  │     
     │    system statistics using psutils to a local file named node_info_<hostname>.txt  and prints that    │     
     │    information to the console.Implement a flag around this code that skips its execution by default   │     
     │    with a comment that the user should toggle the flag in order to run it.Then, create a Slurm job    │     
     │    script that runs this Python code across 2 nodes (1 task per node), requesting GPUs appropriately. │     
     │    After all tasks finish, combine all the generated output files into one file named                 │     
     │    combined_node_info.txt.                                                                            │     
     │  • Overall Goal: Create two files: a Python script (collect_node_info.py) that collects system        │     
     │    metrics with a toggle flag, and a Slurm script (run_nodes.slurm) to run across 2 nodes with GPU    │     
     │    allocation and combine results. All files will be written to a writable directory.                 │     
     │  • Plan Description: We will create two separate files directly: (1) A clean Python script that       │     
     │    collects system stats using psutil, writes to hostname-specific files, prints to console, and      │     
     │    includes a RUN_COLLECTION toggle flag with clear user instructions. (2) A Slurm script that        │     
     │    allocates 2 nodes with 1 task each and 1 GPU per task, runs the Python script on each node, then   │     
     │    concatenates all node_info_*.txt into combined_node_info.txt.                                      │     
     │  • Number of Steps: 2                                                                                 │     
     │                                                                                                       │     
     │ Current Plan Step [2/2]:                                                                              │     
     │                                                                                                       │     
     │  • Step Name: Generate Slurm job script                                                               │     
     │  • Step Description: Generate a Slurm batch script 'run_nodes.slurm' that runs the Python script on 2 │     
     │    nodes with 1 task per node and 1 GPU per task, then combines output files into                     │     
     │    combined_node_info.txt.                            

╭─────────────────────────────────────── Generated Slurm Script ────────────────────────────────────────╮          
│                                               Thoughts                                                │          
│                                                                                                       │          
│ I need to create a Slurm batch script that runs the Python script on 2 nodes with 1 task per node and │          
│ 1 GPU per task, then combines the output files. According to the user task, we're using the Vista     │          
│ system at TACC, and based on the requirements for GPU allocation, the appropriate partition should be │          
│ 'gh' (which provides one GPU per node). I'll set a reasonable time limit of 10 minutes as specified   │          
│ in the user's request. The script will use srun to run the Python script on all nodes in parallel,    │          
│ and then cat all the node_info_*.txt files into a combined file.                                      │          
│                                                                                                       │          
│                                             Slurm Script                                              │          
│                                                                                                       │          
│ Saved to: /home1/02679/agupta/tutorial_spring_26/TACC_exAI/generated_slurm/run_nodes.slurm            │          
│                                                                                                       │          
│                                                                                                       │          
│  #!/bin/bash                                                                                          │          
│  #                                                                                                    │          
│  #SBATCH --job-name=node_info_collection                                                              │          
│  #SBATCH --partition=gh                                                                               │          
│  #SBATCH --time=00:10:00                                                                              │          
│  #SBATCH --nodes=2                                                                                    │          
│  #SBATCH --ntasks-per-node=1                                                                          │          
│  #SBATCH --output=node_info_collection.%j.out                                                         │          
│                                                                                                       │          
│  #!/bin/bash                                                                                          │          
│  #                                                                                                    │          
│  # Slurm batch script to run system information collection on multiple nodes                          │          
│  # and combine results                                                                                │          
│  #                                                                                                    │          
│                                                                                                       │          
│  # Request 2 nodes with 1 task per node (2 tasks total)                                               │          
│  # Request 1 GPU per task                                                                             │          
│  #SBATCH --nodes=2                                                                                    │          
│  #SBATCH --ntasks-per-node=1                                                                          │          
│  #SBATCH --gpus-per-task=1                            

╭─────────────────────────────────────────── Plan Execution Completed ────────────────────────────────────────────╮
│ ✅ Plan execution completed.                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Agent response:
Here is the generated Slurm script for the user's task:

File: /home1/02679/agupta/tutorial_spring_26/TACC_exAI/generated_slurm/run_nodes.slurm

```bash
#!/bin/bash
#
#SBATCH --job-name=node_info_collection
#SBATCH --partition=gh
#SBATCH --time=00:10:00
#SBATCH --nodes=2
#SBATCH --ntasks-per-node=1
#SBATCH --output=node_info_collection.%j.out

#!/bin/bash
#
# Slurm batch script to run system information collection on multiple nodes
# and combine results
#

# Request 2 nodes with 1 task per node (2 tasks total)
# Request 1 GPU per task
#SBATCH --nodes=2
#SBATCH --ntasks-per-node=1
#SBATCH --gpus-per-task=1
#SBATCH --time=00:10:00
#SBATCH --job-name="node_info_collection"
#SBATCH --output=node_info_%A_%a.out

# Print start information
echo "Starting node information collection job at $(date)"
echo "Running on nodes: $SLURM_NODELIST"
echo "Total nodes: $SLURM_NNODES"
echo "Total tasks: $SLURM_NTASKS"

# Clear any previous output files from this run
rm -f node_info_*.txt



## Slurm Script Chaining for Multi-Job Workflows

In this section, we task our agent to handle a more complex, multi‑stage workflow that combines code authoring, Slurm script generation, and iterative job submission.  

We will instruct the agent to:

1. **Write a Python script** that computes digits of π to high precision using the Chudnovsky algorithm (or similar).  
   - The script will store results in a file, e.g. `pi_digits.txt`.  
   - On each run, it will detect how many digits are already present and extend the file by another 10,000 digits.  
   - After completion, it will report how many digits are now stored.

2. **Create a Slurm job submission script** to run this computation efficiently on an HPC system.  
   - The job requests reasonable CPU, memory, and walltime resources.  
   - All standard output and error streams will be captured in log files (e.g., `pi_job.out`, `pi_job.err`).

3. **Write a Python “launch” script** that uses `sbatch` to queue jobs iteratively.  
   - Each job will sit in the queue, but begin only after the previous one finishes (using Slurm’s dependency feature).  
   - This allows automated, chained computation expansions (e.g., 10,000 → 20,000 → 30,000 digits) that are common in model training and simulation workflows.  
   - Because we plan to test and launch this script manually, we ask the agent to bypass its normal code execution test. To achieve this, we wrap the runnable logic in a **guarded flag block**—defaulting to `True`—which users can later toggle to enable actual execution.

The code cell below initializes the agent, configures its available toolset and planning pipeline, and then defines a detailed prompt specifying the desired behavior. Once executed, the agent will design and output all three scripts—each adapted for HPC use via Slurm.

> **Tip:** The final “launch” script demonstrates how to **chain Slurm jobs automatically** to build cumulative results without manual re‑submission—a powerful automation pattern for iterative HPC workloads.

In [6]:
import sys
import os

# Add the agent framework code to our system path so we can import it
notebook_dir = os.getcwd()
notebook_dir = os.path.join(notebook_dir, "TACC_exAI")
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

# import agent framework
from TACC_exAI.agent import Agent
from TACC_exAI.actions.summarize_and_reply import SummarizeAndReplyAction
from TACC_exAI.actions.create_plan import CreatePlanAction
from TACC_exAI.actions.run_plan import RunPlanAction
from TACC_exAI.actions.generate_code import GenerateCodeAction

from TACC_exAI.actions.generate_slurm_script import GenerateSlurmScriptAction


# Initialize the agent
agent = Agent(
    experiment=True,
    experiment_prompt=None,
    #force_ollama=False,
    force_ollama=True,
    #default_action_model_name="DeepSeek-V3-0324",  # long context model
    default_action_model_name="qwen3-coder-next",  # long context model
    isolated_session=True,
    no_save=True,
    display_mode="light",
    mode="dev",
    use_apptainer=True
    
)

# Configure toolset: enable code generation and slurm generation
agent.available_actions = [
    GenerateCodeAction(agent=agent),
    GenerateSlurmScriptAction(agent=agent),
    SummarizeAndReplyAction(agent=agent),
]

# Configure pipeline: plan creation and plan execution
agent.actions = [
    CreatePlanAction(agent=agent),
    RunPlanAction(agent=agent),
]

# Set the experiment prompt

agent.experiment_prompt = (
    f"Write three scripts:\n"
    f"1. **A Python script** that computes pi to 10,000 digits (using a high‑precision method such as "
    f"Chudnovsky), reads any previously computed digits from a file (e.g., `pi_digits.txt`) if it exists,"
    f" extends the total by 10,000 digits beyond what is already stored, saves the updated digits back "
    f"to the file, and prints how many digits are now stored before exiting."
    f" Place this code under a flag that defaults to skipping its execution.\n"
    f"2. **A Slurm script** that submits this Python script as a job, requests reasonable CPU, memory, "
    f"and walltime, and writes stdout and stderr to log files (e.g., `pi_job.out` and `pi_job.err`), "
    f" exiting when the Python script finishes.\n"
    f"3. **A Python launch script** that submits the Slurm script **iteratively** using sbatch to queue "
    f"successive runs so that each job waits in the slurm queue but starts only after the previous job "
    f" finishes, adding 10,000 more digits of pi with each run (from 10,000 → 20,000 → 30,000), without "
    f"checking the file contents; assume each run simply appends 10,000 more digits on top of the prior "
    f"result. Use sbatch features to accomplish the slurm script chaining.\n"
    f"Note: Write the final script with a flag default to true that skips the whole script and tells "
    f"the user to edit the script to flip it to false. Within the skipped section, give the script "
    f"a CLI to ingest the path to the slurm and python scripts, I'll run it later."
)
# Run the agent and capture output
response = agent.run()
print("\nAgent response:")
print(response)

╭──────────────────────────────────────────────── System Message ─────────────────────────────────────────────────╮
│ ⚠️ Forcing local Ollama backend                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Vector Store Status ──────────────────────────────────────────────╮
│ ⚠️ No session specified and no existing vector store found. Running without history vector store.                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Container Backend ───────────────────────────────────────────────╮
│ Using Apptainer for code execution                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Assistant ─────────────────────────────────────────╮                    
│ 👋 How can I assist you today?                                                              │                    
╰─────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────── Prompt to LLM for DynamicPlanForLLM ─────────────────────────────────╮     
     │ Using model qwen3-coder-next on backend Local Ollama.                                                 │     
     │                                                                                                       │     
     │ Prompt:                                                                                               │     
     │                                                                                                       │     
     │ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │     
     │ ┃                                templates/create_plan_template.txt                                 ┃ │     
     │ ┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛ │     
     │                                                                                                       │     
     │ The user has requested: Write three scripts:                                                          │     
     │                                                                                                       │     
     │  1 A Python script that computes pi to 10,000 digits (using a high‑precision method such as           │     
     │    Chudnovsky), reads any previously computed digits from a file (e.g., pi_digits.txt) if it exists,  │     
     │    extends the total by 10,000 digits beyond what is already stored, saves the updated digits back to │     
     │    the file, and prints how many digits are now stored before exiting. Place this code under a flag   │     
     │    that defaults to skipping its execution.                                                           │     
     │  2 A Slurm script that submits this Python script as a job, requests reasonable CPU, memory, and      │     
     │    walltime, and writes stdout and stderr to log files (e.g., pi_job.out and pi_job.err),  exiting    │     
     │    when the Python script finishes.                                                                   │     
     │  3 A Python launch script that submits the Slurm script iteratively using sbatch to queue successive  │     
     │    runs so that each job waits in the slurm queue but starts only after the previous job  finishes,   │     
     │    adding 10,000 more digits of pi with each run (from 10,000 → 20,000 → 30,000), without checking    │     
     │    the file contents; assume each run simply appends 10,000 more digits on top of the prior result.   │     
     │    Use sbatch features to accomplish the slurm script chaining. Note: Write the final script with a   │     
     │    flag default to true that skips the whole script and tells the user to edit the script to flip it  │     
     │    to false. Within the skipped section, give the script a CLI to ingest the path to the slurm and    │     
     │    python scripts, I'll run it later.                                                                 │     
     │                                                                                                       │     
     │ Your task is to create a plan to solve the user's request using the available actions.                │     
     │                                                                                                       │     
     │ Available Actions: Enum DynamicEnum members: GenerateCodeAction: GenerateCodeAction — Generate Python │     
     │ code for user task and execute it inside a Docker container. Returns only the standard output or      │     
     │ error from execution.  Always try and write all of the code in one shot that you can.                 │     
     │ GenerateSlurmScriptAction: GenerateSlurmScriptAction — Generate a Slurm submit script for the user's  │     
     │ task and save it to the local directory. SummarizeAndR

...

╭─────────────────────────────────────── Plan Execution Progress ───────────────────────────────────────╮     
     │ Plan Progress: 3/3                                                                                    │     
     │                                                                                                       │     
     │ Current Action: GenerateCodeAction                                                                    │     
     │                                                                                                       │     
     │ Step Instructions: Create a Python script named 'launch_pi_chain.py' with this behavior:              │     
     │                                                                                                       │     
     │                                                                                                       │     
     │  • Has a SKIP flag at the top defaulting to True. If True, prints a message: 'Skipping execution.     │     
     │    Edit this script to set SKIP=False to enable it.', then exits.                                     │     
     │  • When SKIP=False, it provides a CLI using argparse with: --slurm-script (default 'pi_slurm.sh')     │     
     │    --python-script (default 'pi_computer.py') --num-jobs (default 3) — how many chained jobs to       │     
     │    submit (e.g., 3 → jobs compute 10k, 20k, 30k digits)                                               │     
     │  • It iterates to submit jobs, but each job dependents on the previous one using sbatch's             │     
     │    --dependency=afterany:. The first job has no dependency.                                           │     
     │  • It stores each returned job ID (from sbatch stdout) and uses it for the next dependency.           │     
     │  • After submitting all jobs, it prints the IDs of all submitted jobs.                                │     
     │  • Does not check or read pi_digits.txt contents; assumes each job extends digits by 10,000 as per    │     
     │    its design.                                                                                        │     
     │  • Uses subprocess.run(["sbatch", "--dependency=afterany:<prev_job_id>", "pi_slurm.sh"],              │     
     │    capture_output=True, text=True) pattern.                                                           │     
     │  • Does not parse job status from Slurm—just manages the job IDs and dependencies.                    │     
     │                                                                                                       │     
     │ Write the script to 'launch_pi_chain.py'.                                                             │     
     ╰───────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────── Prompt to LLM for GenerateCode ────────────────────────────────────╮     
     │ Using model qwen3-coder-next on backend Local Ollama.                                                 │     
     │                                                                                                       │     
     │ Prompt: The user has asked to generate Python code for the following task: Full Plan:                 │     
     │                                                                                                       │     
     │  • User Request: Write three scripts:                                                                 │     
     │                                                                                                       │     
     │  1 A Python script that computes pi to 10,000 digits (using a high‑precision method such as           │     
     │    Chudnovsky), reads any previously computed digits from a file (e.g., pi_digits.txt) if it exists,  │     
     │    extends the total by 10,000 digits beyond what is already stored, saves the updated digits back to │     
     │    the file, and prints how many digits are now stored before exiting. Place this code under a flag   │     
     │    that defaults to skipping its execution.                                                           │     
     │  2 A Slurm script that submits this Python script as a job, requests reasonable CPU, memory, and      │     
     │    walltime, and writes stdout and stderr to log files (e.g., pi_job.out and pi_job.err),  exiting    │     
     │    when the Python script finishes.                                                                   │     
     │  3 A Python launch script that submits the Slurm script iteratively using sbatch to queue successive  │     
     │    runs so that each job waits in the slurm queue but starts only after the previous job  finishes,   │     
     │    adding 10,000 more digits of pi with each run (from 10,000 → 20,000 → 30,000), without checking    │     
     │    the file contents; assume each run simply appends 10,000 more digits on top of the prior result.   │     
     │    Use sbatch features to accomplish the slurm script chaining. Note: Write the final script with a   │     
     │    flag default to true that skips the whole script and tells the user to edit the script to flip it  │     
     │    to false. Within the skipped section, give the script a CLI to ingest the path to the slurm and    │     
     │    python scripts, I'll run it later.                                                                 │     
     │                                                                                                       │     
     │  • Overall Goal: Create and validate three coordinated scripts: (1) a Python script that              │     
     │    incrementally computes pi digits using Chudnovsky and persistent storage, (2) a Slurm script to    │     
     │    run one instance of the Python script, and (3) a Python launch script that chains Slurm jobs       │     
     │    iteratively using sbatch dependencies, all with safe default guards and CLI support.               │     
     │  • Plan Description: The plan will: (1) Generate the Python script that computes pi using mpmath with │     
     │    Chudnovsky, saves/loads digits to/from pi_digits.txt, adds 10k digits per run, and respects a skip │     
     │    flag; (2) Generate a Slurm script with reasonable resource requests and output redirections for    │     
     │    the Python job; (3) Generate a Python launch script that, when activated, submits a chain of Slurm │     
     │    jobs via sbatch with dependency chaining. For validation and demonstration, execute the Python     │     
     │    script once to ensure basic functionality and correctness (e.g., compare first digits), then test  │     
     │    the Slurm script locally to ensure it can be genera

╭────────────────────────────────────── Generating Code Attempt 1 ──────────────────────────────────────╮          
│                                               Thoughts                                                │          
│                                                                                                       │          
│ The user request involves creating three interrelated scripts for a high-precision, iterative         │          
│ computation of pi using Slurm job chaining. Key challenges include ensuring the Chudnovsky algorithm  │          
│ is implemented with arbitrary precision (e.g., using mpmath), properly handling file-based digit      │          
│ persistence (with careful string slicing to avoid truncating or duplicating digits), and using        │          
│ Slurm's dependency mechanism ( --dependency=afterany ) for chaining. Additionally, the Python launch  │          
│ script must default to skipping execution and provide a CLI interface, which is straightforward with  │          
│ argparse. All components need to be robust, especially file I/O (append vs overwrite) and ensuring    │          
│ the Slurm scripts are correctly parameterized (time, memory, etc.).                                   │          
│                                                                                                       │          
│                                                 Code                                                  │          
│                                                                                                       │          
│                                                                                                       │          
│  import sys                                                                                           │          
│  import subprocess                                                                                    │          
│                                                                                                       │          
│  # no other imports will work here, they must be first installed and imported in the main function    │          
│  below!                                                                                               │          
│                                                                                                       │          
│  def install_uv():                                                                                    │          
│      try:                                                                                             │          
│          subprocess.check_call([sys.executable, "-m", "pip", "install", "uv"],                        │          
│  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)                                                │          
│      except subprocess.CalledProcessError as e:                                                       │          
│          print(f"Failed to install uv: {e}")                                                          │          
│          sys.exit(1)                                                                                  │          
│                                                                                                       │          
│  def install_packages_with_uv(required_packages):                                                     │          
│      if not required_packages:                                                                        │          
│          return                                                                                       │          
│      try:                                                                                             │          
│          subprocess.check_call([sys.executable, "-m", "uv", "pip", "install"] + required_packages,    │          
│  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

╭─────────────────────────────────────────── Plan Execution Completed ────────────────────────────────────────────╮
│ ✅ Plan execution completed.                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Agent response:
Here is the generated code that solved the user's task:

```python
import sys
import subprocess

# no other imports will work here, they must be first installed and imported in the main function below!

def install_uv():
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "uv"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError as e:
        print(f"Failed to install uv: {e}")
        sys.exit(1)

def install_packages_with_uv(required_packages):
    if not required_packages:
        return
    try:
        subprocess.check_call([sys.executable, "-m", "uv", "pip", "install"] + required_packages, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError as e:
        print(f"Failed to install packages with uv: {e}")
        sys.exit(1)

def main():
    required_packages = ["mpmath"]
    
    if required_packages:
        install_uv()
        install_packages_with

# Take home exercises

1. Examine and Try manually running the scripts generated in these last 2 examples to verify their functionality
    -  **Note:** There will be the following modifications required:
        -  For the **Python scripts** :
            -  You will need to toggle the execution flag variable if present.
            -  You should run the python scripts on a compute node and **NOT** on the login node.
            -  Get a compute node session on command line using the "idev" command **OR**
            -  Within the notebook session, navigate to "File" -> "New" -> "Terminal" and go the the new terminal tab
        -  For the **SLURM scripts** *(**AND** for any Python code that is invoking a slurm command):  
            -  These cannot be run on a compute node and should be run on a login node
                -  Obtain an ssh session to a Vista login node via a command line terminal
                -  Again for the python code, you will need to toggle the execution flag variable
                -  For SLURM scripts, you will need to add further information in order to get it to work:   
                    -  Your Project Account (-A " " parameter to sbatch)  
                    -  Your Reservation Details (--reservation " " parameter to sbatch)
                        - On a login node commandline session run `scontrol show reservations` to see the name of the reservation you are listed on.



2. Prompt Alteration Exercise:
   Alter the prompt in the last example to create a single job with multiple jobsteps, each running a different instance of the pi calculation.  
   Each jobstep should output to a different file. The final jobstep should parse all the output files of the pi calculation and calulate the average of all values
   and output to the console.

   
   